<h1>DWH<h1>

Imports

In [1]:
import pandas as pd
import numpy as np
import sqlite3

Logging creëeren

In [2]:
import logging
import os
os.makedirs("logs", exist_ok=True)

logging.basicConfig( 
    filename="logs/dwh_de.log",
    level=logging.INFO, 
    format="%(asctime)s | %(name)s |  %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H-%M-%S",
    force=True
    )

logger = logging.getLogger("DWH")
logger.setLevel(logging.INFO)

Data connecties

In [3]:
#sdm connectie
sdm_conn = sqlite3.connect("Data/sdmdb.db")
sdm_cursor = sdm_conn.cursor()

#dwh connectie
dwh_conn = sqlite3.connect("Data/dwhdb.db")
dwh_cursor = dwh_conn.cursor()

DWH creëeren

In [36]:
dwh_sql_script = """
CREATE TABLE DIM_customer_segment (
    SEGMENT_CODE INT PRIMARY KEY,
    LANGUAGE VARCHAR(10),
    SEGMENT_NAME VARCHAR(45),
    SEGMENT_DESCRIPTION VARCHAR(100)
);

CREATE TABLE DIM_country (
    COUNTRY_CODE INT PRIMARY KEY,
    COUNTRY VARCHAR(45),
    LANGUAGE VARCHAR(10),
    CURRENCY VARCHAR(45),
    FLAG_IMAGE VARCHAR(100),
    TERRITORY_NAME_EN VARCHAR(45)
);

CREATE TABLE DIM_customer_headquarters (
    CUSTOMER_CODE_HQ INT PRIMARY KEY,
    CUSTOMER_NAME VARCHAR(100),
    ADRESS1 VARCHAR(100),
    ADRESS2 VARCHAR(100),
    CITY VARCHAR(45),
    REGION VARCHAR(45),
    POSTAL_ZONE VARCHAR(45),
    PHONE VARCHAR(45),
    FAX VARCHAR(45),
    COUNTRY INT,
    SEGMENT_CODE INT,
    FOREIGN KEY (COUNTRY) REFERENCES DIM_country(COUNTRY_CODE),
    FOREIGN KEY (SEGMENT_CODE) REFERENCES DIM_customer_segment(SEGMENT_CODE)
);

CREATE TABLE DIM_customer (
    CUSTOMER_CODE INT PRIMARY KEY,
    COMPANY_NAME VARCHAR(100),
    CUSTOMER_TYPE_EN VARCHAR(100),
    CUSTOMER_HEADQUARTERS INT,
    FOREIGN KEY (CUSTOMER_HEADQUARTERS) REFERENCES DIM_customer_headquarters(CUSTOMER_CODE_HQ)
);

CREATE TABLE FACT_sales_demographic(
    DEMOGRAPHIC_CODE INT PRIMARY KEY,
    SALES_PERCENT INT,
    CUSTOMER_HEADQUARTERS INT,
    AGE_GROUP VARCHAR(45),
    FOREIGN KEY (CUSTOMER_HEADQUARTERS) REFERENCES DIM_customer_headquarters(CUSTOMER_CODE_HQ)
);

CREATE TABLE DIM_product(
    PRODUCT_NUMBER INT PRIMARY KEY,
    DESCRIPTION VARCHAR(100),
    INTRODUCTION_DATE DATE,
    COST_CATEGORY VARCHAR(45),
    MARGIN_CATEGORY VARCHAR(45),
    PRODUCT_IMG VARCHAR(45),
    LANGUAGE VARCHAR(10),
    PRODUCT_NAME VARCHAR(100),
    PRODUCT_TYPE_EN VARCHAR(100),
    PRODUCT_LINE_EN VARCHAR(100)
);

CREATE TABLE DIM_retailer_site(
    RETAILER_SITE_CODE INT PRIMARY KEY,
    RETAILER_CODE INT,
    ADRESS1 VARCHAR(100),
    ADRESS2 VARCHAR(100),
    CITY VARCHAR(100),
    REGION VARCHAR(100),
    POSTAL_CODE VARCHAR(45),
    ACTIVE_INDICATOR BOOLEAN,
    COUNTRY INT,
    CUSTOMER INT,
    FOREIGN KEY (CUSTOMER) REFERENCES DIM_customer(CUSTOMER_CODE),
    FOREIGN KEY (COUNTRY) REFERENCES DIM_country(COUNTRY_CODE)
);

CREATE TABLE DIM_sales_branch(
    SALES_BRANCH_CODE INT PRIMARY KEY,
    ADRESS1 VARCHAR(100),
    ADRESS2 VARCHAR(100),
    CITY VARCHAR(100),
    REGION VARCHAR(100),
    POSTAL_CODE VARCHAR(45),
    COUNTRY INT,
    FOREIGN KEY (COUNTRY) REFERENCES DIM_country(COUNTRY_CODE)
);

CREATE TABLE DIM_sales_staff(
    SALES_STAFF_CODE INT PRIMARY KEY,
    FIRST_NAME VARCHAR(45),
    LAST_NAME VARCHAR(45),
    POSITION_EN VARCHAR(100),
    WORK_PHONE VARCHAR(45),
    EXTENSION INT,
    FAX VARCHAR(45),
    DATE_HIRED DATE,
    TRAINING_YEAR INT,
    TRAINING_COURSE VARCHAR(100),
    STATISFACTION_YEAR INT,
    STATISFACTION_DESCRIPTION VARCHAR(100),
    SALES_BRANCH INT,
    MANAGER INT,
    FOREIGN KEY (SALES_BRANCH) REFERENCES DIM_sales_branch(SALES_BRANCH_CODE),
    FOREIGN KEY (MANAGER) REFERENCES DIM_sales_staff(SALES_STAFF_CODE)
);

CREATE TABLE FACT_order_header(
    ORDER_NUMBER INT PRIMARY KEY,
    RETAILER_NAME VARCHAR(100),
    RETAILER_CONTACT_CODE INT,
    ORDER_DATE DATE,
    ORDER_METHOD_EN VARCHAR(100),
    RETAILER_SITE INT,
    SALES_STAFF INT,
    SALES_BRANCH INT,
    FOREIGN KEY (RETAILER_SITE) REFERENCES DIM_retailer_site(RETAILER_SITE_CODE),
    FOREIGN KEY (SALES_STAFF) REFERENCES DIM_sales_staff(SALES_STAFF_CODE),
    FOREIGN KEY (SALES_BRANCH) REFERENCES DIM_sales_branch(SALES_BRANCH_CODE)
);

CREATE TABLE FACT_order_details(
    ORDER_DETAIL_CODE INT PRIMARY KEY,
    QUANTITY INT,
    UNIT_COST DECIMAL(10,2),
    UNIT_REVENUE DECIMAL(10,2),
    UNIT_PRICE DECIMAL(10,2),
    UNIT_SALE_PRICE DECIMAL(10,2),
    ORDER_HEADER INT,
    PRODUCT INT,
    FOREIGN KEY (ORDER_HEADER) REFERENCES FACT_order_header(ORDER_NUMBER),
    FOREIGN KEY (PRODUCT) REFERENCES DIM_product(PRODUCT_NUMBER)
);

CREATE TABLE FACT_return_details(
    RETURN_CODE INT PRIMARY KEY,
    RETURN_DATE DATE,
    QUANTITY VARCHAR(100),
    RETURN_REASON_EN VARCHAR(100),
    ORDER_DETAILS INT,
    FOREIGN KEY (ORDER_DETAILS) REFERENCES FACT_order_details(ORDER_DETAIL_CODE)
);

CREATE TABLE FACT_sales_target(
    SALES_TARGET_CODE INT PRIMARY KEY,
    SALES_YEAR INT,
    SALES_PERIOD INT,
    RETAILER_NAME VARCHAR(100),
    RETAILER_CODE INT,
    SALES_TARGET DECIMAL(10,2),
    SALES_STAFF INT,
    PRODUCT INT,
    FOREIGN KEY (SALES_STAFF) REFERENCES DIM_sales_staff(SALES_STAFF_CODE),
    FOREIGN KEY (PRODUCT) REFERENCES DIM_product(PRODUCT_NUMBER)
);

CREATE TABLE DIM_Month(
    MONTH_ID INT PRIMARY KEY,
    MONTHNR INT,
    MONTH VARCHAR(45),
    QUARTER INT,
    YEAR INT
);

CREATE TABLE DIM_Date(
    DATE_ID INT PRIMARY KEY,
    DATUM DATE,
    MONTH INT,
    FOREIGN KEY (MONTH) REFERENCES DIM_Month(MONTH_ID)
);

CREATE TABLE FACT_Product_Snapshot(
    PRODUCT_SNAPSHOT_CODE INT PRIMARY KEY,
    INVENTORY_COUNT INT,
    EXPECTED_VOLUME INT,
    PRODUCT_NUMBER INT,
    MONTH INT,
    FOREIGN KEY (PRODUCT_NUMBER) REFERENCES DIM_product(PRODUCT_NUMBER),
    FOREIGN KEY (MONTH) REFERENCES DIM_Month(MONTH_ID)
);
"""

dwh_cursor.executescript(dwh_sql_script)
dwh_conn.commit()

SDM uitlezen

In [4]:
age_group = pd.read_sql("SELECT * FROM age_group;",sdm_conn)
customer_contact = pd.read_sql("SELECT * FROM customer_contact;",sdm_conn)
customer_segment = pd.read_sql("SELECT * FROM customer_segment;",sdm_conn)
customer_type = pd.read_sql("SELECT * FROM customer_type;",sdm_conn)
sales_territory = pd.read_sql("SELECT * FROM sales_territory;",sdm_conn)
customer_store = pd.read_sql("SELECT * FROM customer_store;",sdm_conn)
crm_country = pd.read_sql("SELECT * FROM crm_country;",sdm_conn)
customer = pd.read_sql("SELECT * FROM customer;",sdm_conn)
sales_demographic = pd.read_sql("SELECT * FROM sales_demographic;",sdm_conn)
customer_headquarters = pd.read_sql("SELECT * FROM customer_headquarters;",sdm_conn)

order_method = pd.read_sql("SELECT * FROM order_method;",sdm_conn)
product_line = pd.read_sql("SELECT * FROM product_line;",sdm_conn)
retailer_site = pd.read_sql("SELECT * FROM retailer_site;",sdm_conn)
return_reason = pd.read_sql("SELECT * FROM return_reason;",sdm_conn)
returned_item = pd.read_sql("SELECT * FROM returned_item;",sdm_conn)
order_details = pd.read_sql("SELECT * FROM order_details;",sdm_conn)
order_header = pd.read_sql("SELECT * FROM order_header;",sdm_conn)
product_type = pd.read_sql("SELECT * FROM product_type;",sdm_conn)
product = pd.read_sql("SELECT * FROM product;",sdm_conn)
sales_staff = pd.read_sql("SELECT * FROM sales_staff;",sdm_conn)
go_sales_country = pd.read_sql("SELECT * FROM go_sales_country;",sdm_conn)
sales_branch = pd.read_sql("SELECT * FROM sales_branch;",sdm_conn)

satisfaction = pd.read_sql("SELECT * FROM satisfaction;",sdm_conn)
training = pd.read_sql("SELECT * FROM training;",sdm_conn)
course = pd.read_sql("SELECT * FROM course;",sdm_conn)
sales_office = pd.read_sql("SELECT * FROM sales_office;",sdm_conn)
satisfaction_type = pd.read_sql("SELECT * FROM satisfaction_type;",sdm_conn)
sales_representative = pd.read_sql("SELECT * FROM sales_representative;",sdm_conn)

inventory_levels = pd.read_sql("SELECT * FROM inventory_level", sdm_conn)
product_forecast = pd.read_sql("SELECT * FROM product_forecast", sdm_conn)
sales_target = pd.read_sql("SELECT * FROM sales_target", sdm_conn)

Lijst DWH tabellen

In [30]:
alle_dwh_tabellen = {
    'Data/dwhdb.db': {
    "DIM_Date" : 'DIM_Date',
    "DIM_Month" : 'DIM_Month',
    "DIM_country" : 'DIM_country',
    "DIM_customer" : 'DIM_customer',
    "DIM_customer_headquarters" : 'DIM_customer_headquarters',
    "DIM_inventory_level" : 'DIM_inventory_level',
    "DIM_product" : 'DIM_product',
    "DIM_product_forecast" : 'DIM_product_forecast',
    "DIM_retailer_site" : 'DIM_retailer_site',
    "DIM_sales_branch" : 'DIM_sales_branch',
    "DIM_sales_staff" : 'DIM_sales_staff',
    "FACT_return_details" : 'FACT_return_details',
    "FACT_order_details" : 'FACT_order_details',
    "FACT_order_header" : 'FACT_order_header',
    "FACT_sales_demographic" : 'FACT_sales_demographic',
    "FACT_sales_target" : 'FACT_sales_target'
    }
 }

DIM_sales_staff

In [6]:
sales_representative = sales_representative.drop(columns='EXTENSION')

DIM_sales_Staff = pd.merge(
    sales_representative,
    sales_staff,
    how='left',
    on=['FIRST_NAME', 'LAST_NAME', 'POSITION_EN',
       'WORK_PHONE', 'FAX', 'DATE_HIRED']
)

DIM_sales_Staff.insert(0, 'SALES_STAFF_CODE', DIM_sales_Staff.pop('SALES_STAFF_CODE'))
DIM_sales_Staff['DATE_HIRED'] = pd.to_datetime(DIM_sales_Staff['DATE_HIRED'], format="%d-%b-%Y %I:%M:%S %p")

training = training.rename(columns={'SALES_REPRESENTATIVE' : 'SALES_REPRESENTATIVE_CODE'})
satisfaction = satisfaction.rename(columns={'SALES_REPRESENTATIVE' : 'SALES_REPRESENTATIVE_CODE'})
satisfaction_type = satisfaction_type.rename(columns={'SATISFACTION_TYPE_CODE' : 'SATISFACTION_TYPE'})

DIM_sales_Staff = DIM_sales_Staff.merge(training, how='left', on='SALES_REPRESENTATIVE_CODE')
DIM_sales_Staff = DIM_sales_Staff.rename(columns={'MANAGER_CODE':'MANAGER', 'YEAR': 'TRAINING_YEAR', 'TRAINING_CODE':'TRAINING_COURSE'})
DIM_sales_Staff = DIM_sales_Staff.merge(satisfaction, how='left', on='SALES_REPRESENTATIVE_CODE')
DIM_sales_Staff = DIM_sales_Staff.merge(satisfaction_type, how='left', on='SATISFACTION_TYPE')
DIM_sales_Staff = DIM_sales_Staff.rename(columns={'YEAR': 'STATISFACTION_YEAR', 'SATISFACTION_TYPE_DESCRIPTION': 'STATISFACTION_DESCRIPTION'})
DIM_sales_Staff['DATE_HIRED'] = DIM_sales_Staff['DATE_HIRED'].astype(str)
DIM_sales_Staff = DIM_sales_Staff.drop_duplicates(
    subset=['SALES_STAFF_CODE']
)

#DIM_sales_Staff

DIM_country

In [7]:
crm_country = crm_country.rename(columns={'COUNTRY_EN':'COUNTRY'})

DIM_Country = pd.merge(
    go_sales_country,
    crm_country,
    how='outer',
    on=['COUNTRY_CODE', 'COUNTRY']
)

DIM_Country = DIM_Country.rename(columns={'SALES_TERRITORY':'SALES_TERRITORY_CODE'})
DIM_Country = pd.merge(
    DIM_Country,
    sales_territory,
    how='left',
    on=['SALES_TERRITORY_CODE']
)

DIM_Country = DIM_Country.drop(columns='SALES_TERRITORY_CODE')
DIM_Country = DIM_Country.rename(columns={'CURRENCY_NAME' : 'CURRENCY'})

#DIM_Country

DIM_sales_branch

In [8]:
sales_office = sales_office.rename(columns={'STREET' : 'ADRESS1', 'ADDITION' : 'ADRESS2','ZIPCODE':'POSTAL_CODE'})

DIM_Sales_Branch = pd.merge(
    sales_branch,
    sales_office,
    how='left',
    on=['ADRESS1','ADRESS2','CITY', 'REGION', 'POSTAL_CODE','COUNTRY']
)

#DIM_Sales_Branch

DIM_Retailer_Site

In [9]:
customer_store = customer_store.rename(columns={'ZIPCODE':'POSTAL_ZONE',
                                                'STATE': 'REGION',
                                                'CRM_COUNTRY' : 'COUNTRY',
                                                'STREET' : 'ADRESS1',
                                                'ADDITION' : 'ADRESS2'})

DIM_Retailer_Site = pd.merge(
    retailer_site,
    customer_store,
    how='left',
    on=['ADRESS1', 'ADRESS2', 'CITY', 'REGION', 'POSTAL_ZONE', 'COUNTRY', 'ACTIVE_INDICATOR']
)

DIM_Retailer_Site = DIM_Retailer_Site.rename(columns={'POSTAL_ZONE' : 'POSTAL_CODE'})

#DIM_Retailer_Site

DIM_Customer

In [10]:
customer_type = customer_type.rename(columns={'CUSTOMER_TYPE_CODE' : 'CUSTOMER_TYPE'})
DIM_customer = customer.merge(customer_type, how='left', on=['CUSTOMER_TYPE'])
DIM_customer = DIM_customer.rename(columns={'CUSTOMER_CODEMR':'CUSTOMER_HEADQUARTERS'})

#DIM_customer

DIM_Month

In [11]:
DIM_Month = pd.DataFrame()

DIM_Month = pd.concat([
    pd.DataFrame({
        'YEAR': inventory_levels['INVENTORY_YEAR'],
        'MONTHNR': inventory_levels['INVENTORY_MONTH']
    }),
    
    pd.DataFrame({
        'YEAR': product_forecast['YEAR'],
        'MONTHNR': product_forecast['MONTH']
    })
]).drop_duplicates().sort_values(['YEAR', 'MONTHNR']).reset_index(drop=True)

DIM_Month['MONTH'] = DIM_Month['MONTHNR'].map({1 : 'Jan', 2 : 'Feb', 3 : 'Mar', 4 : 'Apr', 5:'Mei', 6 : 'Jun', 7 : 'Jul', 8 : 'Aug', 9 : 'Sep', 10 : 'Oct', 11 : 'Nov', 12 : 'Dec'})
DIM_Month['QUARTER'] = DIM_Month['MONTHNR'].map({ 1: 1, 2 : 1, 3 : 1, 4 : 2, 5:2, 6 : 2, 7 : 3, 8 : 3, 9 : 3, 10 : 4, 11 : 4, 12 : 4})
DIM_Month['MONTH_ID'] = DIM_Month.index

#DIM_Month

DIM_Datum

In [12]:
DIM_Date = pd.DataFrame()

DIM_Date['DATUM'] = pd.concat([
    pd.to_datetime(returned_item['RETURN_DATE'], format="%d-%b-%Y %I:%M:%S %p"),
    pd.to_datetime(product['INTRODUCTION_DATE'], format="%d-%b-%Y %I:%M:%S %p"),
    pd.to_datetime(order_header['ORDER_DATE'], format="%d-%b-%Y %I:%M:%S %p"),
    pd.to_datetime(sales_staff['DATE_HIRED'], format="%d-%b-%Y %I:%M:%S %p")
]).drop_duplicates().sort_values().reset_index(drop=True)
DIM_Date['DAY'] = pd.to_datetime(DIM_Date['DATUM']).dt.day

DIM_Date['YEAR'] = DIM_Date['DATUM'].dt.year
DIM_Date['MONTHNR'] = DIM_Date['DATUM'].dt.month

DIM_Date = DIM_Date.merge(
    DIM_Month[['YEAR', 'MONTHNR', 'MONTH_ID']],
    on=['YEAR', 'MONTHNR'],
    how='left'
)
DIM_Date = DIM_Date.rename(columns={'MONTHNR' : 'MONTH'})
DIM_Date['DATUM'] = DIM_Date['DATUM'].astype(str)
DIM_Date['DATE_ID'] = DIM_Date.index

#DIM_Date

DIM_Product

In [13]:
product = product.rename(columns={'PRODUCT_TYPE' : 'PRODUCT_TYPE_CODE'})

DIM_product = product.merge(product_type, how='left', on='PRODUCT_TYPE_CODE')
DIM_product = DIM_product.rename(columns={'PRODUCT_LINE' : 'PRODUCT_LINE_CODE'})

DIM_product = DIM_product.merge(product_line, how='left', on='PRODUCT_LINE_CODE')
DIM_product['INTRODUCTION_DATE'] = DIM_product['INTRODUCTION_DATE'].astype(str)

q1 = DIM_product['PRODUCTION_COST'].quantile(0.25)
q3 = DIM_product['PRODUCTION_COST'].quantile(0.75)

DIM_product['PRODUCTION_COST'] = DIM_product['PRODUCTION_COST'].astype(float)
DIM_product['COST_CATEGORY'] = 'Gemiddeld'
DIM_product.loc[DIM_product['PRODUCTION_COST'] >= q3,'COST_CATEGORY'] = 'Duur'
DIM_product.loc[DIM_product['PRODUCTION_COST'] <= q1,'COST_CATEGORY'] = 'Goedkoop'

q2 = DIM_product['MARGIN'].quantile(0.25)
q4 = DIM_product['MARGIN'].quantile(0.75)

DIM_product['MARGIN_CATEGORY'] = 'gemiddeld'
DIM_product.loc[DIM_product['MARGIN'] >= q4, 'MARGIN_CATEGORY'] = 'Duur'
DIM_product.loc[DIM_product['MARGIN'] <= q2, 'MARGIN_CATEGORY'] = 'Goedkoop'

DIM_product = DIM_product.rename(columns={'PRODUCT_IMAGE':'PRODUCT_IMG'})

 
#DIM_product

FACT_return_details

In [14]:
returned_item = returned_item.rename(columns={'RETURN_REASON' : 'RETURN_REASON_CODE'})
return_reason = return_reason.rename(columns={'RETURN_DESCRIPTION' : 'RETURN_REASON_EN'})

FACT_return_details = returned_item.merge(return_reason,on='RETURN_REASON_CODE',how='left')

FACT_return_details = FACT_return_details.rename(columns={'ORDER_DETAIL_CODE':'ORDER_DETAILS', 'RETURN_QUANTITY':'QUANTITY', 'RETURN_DESCRIPTION_EN':'RETURN_REASON_EN'})

#FACT_return_details

FACT_Order_Header

In [15]:
order_method = order_method.rename(columns={'ORDER_METHOD_CODE' : 'ORDER_METHOD'})

FACT_order_header = order_header.merge(order_method, how='left', on='ORDER_METHOD')
FACT_order_header = FACT_order_header.rename(columns={'SALES_STAFF_CODE':'SALES_STAFF', 'RETAILER_SITE_CODE':'RETAILER_SITE', 'SALES_BRANCH_CODE':'SALES_BRANCH'})
FACT_order_header['ORDER_DATE'] = FACT_order_header['ORDER_DATE'].astype(str)

#FACT_order_header

FACT_Order_Details

In [16]:
FACT_order_details = order_details.copy()

FACT_order_details['UNIT_PRICE'] = FACT_order_details['UNIT_PRICE'].astype('double')
FACT_order_details['UNIT_COST'] = FACT_order_details['UNIT_COST'].astype('double')
FACT_order_details['UNIT_REVENUE'] = (FACT_order_details['UNIT_PRICE'] - FACT_order_details['UNIT_COST'])

#FACT_order_details

FACT_sales_demographic

In [17]:
age_group['AGE_GROUP'] = (age_group['LOWER_AGE'].astype(str) + '-' + age_group['UPPER_AGE'].astype(str))
sales_demographic = sales_demographic.rename(columns={'AGE_GROUP' : 'AGE_GROUP_CODE'})

FACT_sales_demographic = sales_demographic.merge(age_group, how='left', on ='AGE_GROUP_CODE')

#FACT_sales_demographic

FACT_product_Snapshot

In [ ]:

FACT_product_snapshot = product_forecast.merge(
    inventory_levels, 
    how='left',
    left_on = ['PRODUCT', 'YEAR', 'MONTH'],
    right_on= ['PRODUCT', 'INVENTORY_YEAR', 'INVENTORY_MONTH'])

FACT_product_snapshot['PRODUCT_SNAPSHOT_CODE'] = FACT_product_snapshot.index

FACT_product_snapshot = FACT_product_snapshot.drop(columns=['INVENTORY_YEAR', 'INVENTORY_MONTH'])


FACT_product_snapshot['PRODUCT_SNAPSHOT_CODE'] = range(1,len(FACT_product_snapshot) + 1)

FACT_product_snapshot = FACT_product_snapshot.rename(columns={'PRODUCT' : 'PRODUCT_NUMBER'})

#FACT_product_snapshot

,PRODUCT_FORECAST_CODE,YEAR,MONTH,EXPECTED_VOLUME,PRODUCT_NUMBER,INVENTORY_LEVELS_CODE,INVENTORY_COUNT,PRODUCT_SNAPSHOT_CODE
0,0,2025,12,383,44,3816,8922,1
1,1,2024,1,80,45,3447,1858,2
2,2,2024,2,51,45,3562,1930,3
3,3,2024,3,214,45,3677,1856,4
4,4,2024,4,300,45,3792,1676,5
...,...,...,...,...,...,...,...,...
3867,3867,2025,8,282,115,1907,6654,3868
3868,3868,2025,9,920,115,2022,5634,3869
3869,3869,2025,10,1081,115,2137,4488,3870
3870,3870,2025,11,398,115,2252,4108,3871


Nulls vullen

In [39]:
lijst_country = 'LANGUAGE', 'CURRENCY', 'FLAG_IMAGE','TERRITORY_NAME_EN'
for i in lijst_country:
    DIM_Country[i] = DIM_Country[i].fillna('-')

max_key = int(DIM_sales_Staff['SALES_STAFF_CODE'].max())

mask = DIM_sales_Staff['SALES_STAFF_CODE'].isnull()

DIM_sales_Staff.loc[mask, 'SALES_STAFF_CODE'] = range(
    max_key + 1,
    max_key + 1 + mask.sum()
)

lijst_sales_staff = 'EXTENSION', 'TRAINING_YEAR', 'TRAINING_COURSE', 'STATISFACTION_YEAR', 'SALES_BRANCH','STATISFACTION_DESCRIPTION'
for i in lijst_sales_staff:
    DIM_sales_Staff[i] = DIM_sales_Staff[i].fillna('-')

DWH inladen

In [40]:

dwh_cursor.execute("PRAGMA foreign_keys = OFF;")

# =========================
# 1. ROOT TABLES
# =========================

root = [
    ("DIM_customer_segment", customer_segment, [
        "SEGMENT_CODE",
        "LANGUAGE",
        "SEGMENT_NAME",
        "SEGMENT_DESCRIPTION"
    ]),

    ("DIM_Month", DIM_Month, [
        "MONTH_ID",
        "MONTHNR",
        "MONTH",
        "QUARTER",
        "YEAR"
    ]),
]

# =========================
# 2. MID LAYER
# =========================

mid = [

    ("DIM_country", DIM_Country, [
        "COUNTRY_CODE",
        "COUNTRY",
        "LANGUAGE",
        "CURRENCY",
        "FLAG_IMAGE",
        "TERRITORY_NAME_EN"
    ]),

    ("DIM_Date", DIM_Date, [
        "DATE_ID",
        "DATUM",
        "MONTH"
    ]),

    ("DIM_sales_branch", DIM_Sales_Branch, [
        "SALES_BRANCH_CODE",
        "ADRESS1",
        "ADRESS2",
        "CITY",
        "REGION",
        "POSTAL_CODE",
        "COUNTRY"
    ]),

    ("DIM_product", DIM_product, [
        "PRODUCT_NUMBER",
        "DESCRIPTION",
        "INTRODUCTION_DATE",
        "COST_CATEGORY",
        "MARGIN_CATEGORY",
        "PRODUCT_IMG",
        "LANGUAGE",
        "PRODUCT_NAME",
        "PRODUCT_TYPE_EN",
        "PRODUCT_LINE_EN"
    ]),
]

# =========================
# 3. THIRD LAYER
# =========================

third = [

    ("DIM_customer_headquarters", customer_headquarters, [
        "CUSTOMER_CODE_HQ",
        "CUSTOMER_NAME",
        "ADRESS1",
        "ADRESS2",
        "CITY",
        "REGION",
        "POSTAL_ZONE",
        "PHONE",
        "FAX",
        "COUNTRY",
        "SEGMENT_CODE"
    ]),

    ("DIM_sales_staff", DIM_sales_Staff, [
        "SALES_STAFF_CODE",
        "FIRST_NAME",
        "LAST_NAME",
        "POSITION_EN",
        "WORK_PHONE",
        "EXTENSION",
        "FAX",
        "DATE_HIRED",
        "TRAINING_YEAR",
        "TRAINING_COURSE",
        "STATISFACTION_YEAR",
        "STATISFACTION_DESCRIPTION",
        "SALES_BRANCH",
        "MANAGER"
    ]),

    ("DIM_customer", DIM_customer, [
        "CUSTOMER_CODE",
        "COMPANY_NAME",
        "CUSTOMER_TYPE_EN",
        "CUSTOMER_HEADQUARTERS"
    ]),

    ("DIM_retailer_site", DIM_Retailer_Site, [
        "RETAILER_SITE_CODE",
        "RETAILER_CODE",
        "ADRESS1",
        "ADRESS2",
        "CITY",
        "REGION",
        "POSTAL_CODE",
        "ACTIVE_INDICATOR",
        "COUNTRY",
        "CUSTOMER"
    ]),
]

# =========================
# 4. FACT TABLES
# =========================

fact = [

    ("FACT_sales_demographic", FACT_sales_demographic, [
        "DEMOGRAPHIC_CODE",
        "SALES_PERCENT",
        "CUSTOMER_HEADQUARTERS",
        "AGE_GROUP"
    ]),

    ("FACT_order_header", FACT_order_header, [
        "ORDER_NUMBER",
        "RETAILER_NAME",
        "RETAILER_CONTACT_CODE",
        "ORDER_DATE",
        "ORDER_METHOD_EN",
        "RETAILER_SITE",
        "SALES_STAFF",
        "SALES_BRANCH"
    ]),

    ("FACT_order_details", FACT_order_details, [
        "ORDER_DETAIL_CODE",
        "QUANTITY",
        "UNIT_COST",
        "UNIT_REVENUE",
        "UNIT_PRICE",
        "UNIT_SALE_PRICE",
        "ORDER_HEADER",
        "PRODUCT"
    ]),

    ("FACT_return_details", FACT_return_details, [
        "RETURN_CODE",
        "RETURN_DATE",
        "QUANTITY",
        "RETURN_REASON_EN",
        "ORDER_DETAILS"
    ]),

    ("FACT_product_snapshot", FACT_product_snapshot, [
        "PRODUCT_SNAPSHOT_CODE",
        "INVENTORY_COUNT",
        "EXPECTED_VOLUME",
        "PRODUCT_NUMBER",
        "MONTH"
    ]),

    ("FACT_sales_target", sales_target, [
        "SALES_TARGET_CODE",
        "SALES_YEAR",
        "SALES_PERIOD",
        "RETAILER_NAME",
        "RETAILER_CODE",
        "SALES_TARGET",
        "SALES_STAFF",
        "PRODUCT"
    ]),
]

# =========================
# ALL TABLES
# =========================

all_tables = root + mid + third + fact

for table_name, df, cols in all_tables:

    missing = set(cols) - set(df.columns)

    if missing:
        print(f"❌ {table_name} mist kolommen: {missing}")
        continue

    df_clean = df[cols]

    query = f"""
    INSERT OR REPLACE INTO {table_name}
    ({",".join(cols)})
    VALUES ({",".join(["?"] * len(cols))})
    """

    try:
        dwh_cursor.executemany(query, df_clean.values.tolist())
        print(f"✔ {table_name}")

    except Exception as e:
        print(f"❌ {table_name} ERROR: {e}")
        break

dwh_conn.commit()

✔ DIM_customer_segment
✔ DIM_Month
✔ DIM_country
✔ DIM_Date
✔ DIM_sales_branch
✔ DIM_product
✔ DIM_customer_headquarters
✔ DIM_sales_staff
✔ DIM_customer
✔ DIM_retailer_site
✔ FACT_sales_demographic
✔ FACT_order_header
✔ FACT_order_details
✔ FACT_return_details
✔ FACT_product_snapshot
✔ FACT_sales_target


DWH resetten

In [35]:
def resetDWH():
    """
    Verwijdert alle tabellen uit het DWH
    + logging
    """

    try:
        logger.info("DWH|RESET_DWH_START|Data/dwhdb.db.db|||||START")

        dwh_cursor.execute(
            "PRAGMA foreign_keys = OFF;"
        )

        tables = dwh_cursor.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type='table'
        """).fetchall()

        if not tables:
            logger.warning("DWH|ERROR|Data/dwhdb.db||||FAILED|%s",str(e))

        for (table,) in tables:

            try:
                dwh_cursor.execute(
                    f"DROP TABLE IF EXISTS {table}"
                )

                logger.info(
                "DWH|READ|Data/dwhdb.db|%s|||SUCCESS",
                table
                )

                print(f"Dropped: {table}")

            except Exception as table_error:

                logger.error(
                f"DWH|ERROR|Data/dwhdb.db.db|%s|||FAILED|%s",
                table,
                str(e)
                )

        dwh_conn.commit()

        logger.info("DWH|LOAD_DWH_END|Data/dwhdb.db|||||END")
        print("DWH volledig gereset (DROP)")

    except Exception as e:

        dwh_conn.rollback()

        logger.error(
            f"logger.error("
            "DWH|ERROR|Data/dwhdb.db||||FAILED|%s",
            str(e)
            )

        print(f"ERROR: {e}")

    finally:

        dwh_conn.execute(
            "PRAGMA foreign_keys = ON;"
        )

        logger.info("DWH|RESET_DWH_END|%s|||||END")


# RUN
resetDWH()

Dropped: DIM_customer_segment
Dropped: DIM_country
Dropped: DIM_customer_headquarters
Dropped: DIM_customer
Dropped: FACT_sales_demographic
Dropped: DIM_product
Dropped: DIM_retailer_site
Dropped: DIM_sales_branch
Dropped: DIM_sales_staff
Dropped: FACT_order_header
Dropped: FACT_order_details
Dropped: FACT_return_details
Dropped: FACT_sales_target
Dropped: DIM_Month
Dropped: DIM_Date
Dropped: FACT_Product_Snapshot
DWH volledig gereset (DROP)
